## `Text Prediction System`


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical

`1. Create a small dataset`

In [2]:
corpus = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "the cat ran on the grass",
    "the dog sat on the floor",
    "the bird flew over the tree",
    "the cat chased the bird",
    "the dog chased the cat",
    "the bird sat on the mat",
    "the sun shines over the park",
    "the moon rises over the tree",
    "the boy loves to play football",
    "the girl loves to read books",
    "the teacher loves to teach students",
    "the student loves to learn new things",
    "machine learning is a type of artificial intelligence",
    "deep learning is a subset of machine learning",
    "natural language processing helps computers understand text",
    "neural networks learn from large amounts of data",
    "the model predicts the next word in the sentence",
    "tokenization splits text into individual words",
]

print(f"Dataset size: {len(corpus)} sentences")
for i, s in enumerate(corpus):
    print(f"  [{i+1:2d}] {s}")

Dataset size: 20 sentences
  [ 1] the cat sat on the mat
  [ 2] the dog ran in the park
  [ 3] the cat ran on the grass
  [ 4] the dog sat on the floor
  [ 5] the bird flew over the tree
  [ 6] the cat chased the bird
  [ 7] the dog chased the cat
  [ 8] the bird sat on the mat
  [ 9] the sun shines over the park
  [10] the moon rises over the tree
  [11] the boy loves to play football
  [12] the girl loves to read books
  [13] the teacher loves to teach students
  [14] the student loves to learn new things
  [15] machine learning is a type of artificial intelligence
  [16] deep learning is a subset of machine learning
  [17] natural language processing helps computers understand text
  [18] neural networks learn from large amounts of data
  [19] the model predicts the next word in the sentence
  [20] tokenization splits text into individual words


`2. Tokenize the text`

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

vocab_size = len(tokenizer.word_index) + 1  # +1 for padding token (index 0)
word_index = tokenizer.word_index

print(f"Vocabulary size (including padding): {vocab_size}")
print(f"\nWord → Index mapping (first 20):")
for word, idx in list(word_index.items())[:20]:
    print(f"  '{word}': {idx}")

Vocabulary size (including padding): 69

Word → Index mapping (first 20):
  'the': 1
  'cat': 2
  'on': 3
  'loves': 4
  'to': 5
  'sat': 6
  'dog': 7
  'bird': 8
  'over': 9
  'learning': 10
  'of': 11
  'mat': 12
  'ran': 13
  'in': 14
  'park': 15
  'tree': 16
  'chased': 17
  'learn': 18
  'machine': 19
  'is': 20


In [4]:
# Build n-gram sequences: for each sentence, produce all prefix→next-word pairs
input_sequences = []
for sentence in corpus:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(token_list)):
        n_gram = token_list[:i + 1]   # e.g. [tok1, tok2], [tok1, tok2, tok3] …
        input_sequences.append(n_gram)

max_seq_len = max(len(s) for s in input_sequences)
print(f"Total n-gram sequences: {len(input_sequences)}")
print(f"Max sequence length:    {max_seq_len}")
print(f"\nFirst 5 raw n-grams:")
for seq in input_sequences[:5]:
    words = [list(word_index.keys())[list(word_index.values()).index(i)] for i in seq]
    print(f"  {seq}  →  {words}")

Total n-gram sequences: 109
Max sequence length:    9

First 5 raw n-grams:
  [1, 2]  →  ['the', 'cat']
  [1, 2, 6]  →  ['the', 'cat', 'sat']
  [1, 2, 6, 3]  →  ['the', 'cat', 'sat', 'on']
  [1, 2, 6, 3, 1]  →  ['the', 'cat', 'sat', 'on', 'the']
  [1, 2, 6, 3, 1, 12]  →  ['the', 'cat', 'sat', 'on', 'the', 'mat']


In [5]:
# Pad sequences and split into features (X) and labels (y)
padded = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = padded[:, :-1]   # all tokens except the last
y = padded[:, -1]    # the last token is the target

y_cat = to_categorical(y, num_classes=vocab_size)

print(f"X shape: {X.shape}")
print(f"y shape: {y_cat.shape}")
print(f"\nSample X[0]:  {X[0]}  →  y label index: {y[0]}")

X shape: (109, 8)
y shape: (109, 69)

Sample X[0]:  [0 0 0 0 0 0 0 1]  →  y label index: 2


`3. Train a neural network (Embedding + LSTM)`

In [6]:
tf.random.set_seed(42)

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_seq_len - 1),
    LSTM(128),
    Dense(vocab_size, activation='softmax'),
])

model.compile(loss='categorical_crossentropy', 
              optimizer='adam', 
              metrics=['accuracy'])
model.summary()

c:\Users\Administrator\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model.fit(X, y_cat, epochs=200, verbose=0)

# Show training progress at key checkpoints
for epoch in [0, 49, 99, 149, 199]:
    print(f"Epoch {epoch+1:3d} — loss: {history.history['loss'][epoch]:.4f}  "
          f"accuracy: {history.history['accuracy'][epoch]:.4f}")

Epoch   1 — loss: 4.2323  accuracy: 0.0642
Epoch  50 — loss: 1.4795  accuracy: 0.5963
Epoch 100 — loss: 0.6595  accuracy: 0.8349
Epoch 150 — loss: 0.4723  accuracy: 0.8440
Epoch 200 — loss: 0.4246  accuracy: 0.8440


`4. Predict the next word`

In [8]:
index_to_word = {idx: word for word, idx in word_index.items()}

def predict_next_word(seed_text, top_n=3):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
    probs = model.predict(token_list, verbose=0)[0]

    top_indices = np.argsort(probs)[::-1][:top_n]
    results = [(index_to_word.get(i, '?'), float(probs[i])) for i in top_indices]
    return results

# Test predictions
test_seeds = [
    "the cat",
    "the dog",
    "machine learning is",
    "deep learning",
    "natural language",
]

print("Next-word predictions:\n")
for seed in test_seeds:
    preds = predict_next_word(seed, top_n=3)
    top_word, top_conf = preds[0]
    others = ", ".join(f"'{w}' ({c:.2%})" for w, c in preds[1:])
    print(f"  Seed: '{seed}'")
    print(f"    → '{top_word}' ({top_conf:.2%})   |   alternatives: {others}")
    print()

Next-word predictions:

  Seed: 'the cat'
    → 'ran' (51.26%)   |   alternatives: 'sat' (23.08%), 'chased' (21.44%)

  Seed: 'the dog'
    → 'ran' (33.17%)   |   alternatives: 'sat' (31.70%), 'chased' (29.30%)

  Seed: 'machine learning is'
    → 'a' (96.06%)   |   alternatives: 'is' (1.77%), 'type' (1.15%)

  Seed: 'deep learning'
    → 'is' (95.43%)   |   alternatives: 'a' (1.67%), 'learning' (0.97%)

  Seed: 'natural language'
    → 'processing' (89.85%)   |   alternatives: 'language' (3.46%), 'helps' (2.09%)



---
### `TensorFlow NLP Program`

This task demonstrates:
1. Accepting arbitrary text input
2. Tokenizing the text
3. Displaying word indices
4. Converting text into integer sequences

`1. Accept text input`

In [9]:
sample_texts = [
    "I love natural language processing",
    "deep learning models are powerful",
    "the quick brown fox jumps over the lazy dog",
    "TensorFlow makes building neural networks easy",
]

print("=== Input Texts ===")
for i, t in enumerate(sample_texts, 1):
    print(f"  [{i}] {t}")

=== Input Texts ===
  [1] I love natural language processing
  [2] deep learning models are powerful
  [3] the quick brown fox jumps over the lazy dog
  [4] TensorFlow makes building neural networks easy


`2. Tokenize the text & display word indices`

In [10]:
task3_tokenizer = Tokenizer(oov_token="<OOV>")
task3_tokenizer.fit_on_texts(sample_texts)

word_index_t3 = task3_tokenizer.word_index
print("=== Word Index (all unique tokens) ===")
print(f"{'Index':>6}  Word")
print("-" * 22)
for word, idx in sorted(word_index_t3.items(), key=lambda x: x[1]):
    print(f"{idx:>6}  {word}")

=== Word Index (all unique tokens) ===
 Index  Word
----------------------
     1  <OOV>
     2  the
     3  i
     4  love
     5  natural
     6  language
     7  processing
     8  deep
     9  learning
    10  models
    11  are
    12  powerful
    13  quick
    14  brown
    15  fox
    16  jumps
    17  over
    18  lazy
    19  dog
    20  tensorflow
    21  makes
    22  building
    23  neural
    24  networks
    25  easy


`3. Convert text into sequences`

In [11]:
sequences_t3 = task3_tokenizer.texts_to_sequences(sample_texts)

print("=== Text → Integer Sequence ===\n")
for text, seq in zip(sample_texts, sequences_t3):
    print(f"  Text    : {text}")
    print(f"  Sequence: {seq}")
    print()

# Padded form (uniform length for feeding into a model)
max_len_t3 = max(len(s) for s in sequences_t3)
padded_t3 = pad_sequences(sequences_t3, maxlen=max_len_t3, padding='post')

print("=== Padded Sequences (post-padding) ===")
print(padded_t3)

=== Text → Integer Sequence ===

  Text    : I love natural language processing
  Sequence: [3, 4, 5, 6, 7]

  Text    : deep learning models are powerful
  Sequence: [8, 9, 10, 11, 12]

  Text    : the quick brown fox jumps over the lazy dog
  Sequence: [2, 13, 14, 15, 16, 17, 2, 18, 19]

  Text    : TensorFlow makes building neural networks easy
  Sequence: [20, 21, 22, 23, 24, 25]

=== Padded Sequences (post-padding) ===
[[ 3  4  5  6  7  0  0  0  0]
 [ 8  9 10 11 12  0  0  0  0]
 [ 2 13 14 15 16 17  2 18 19]
 [20 21 22 23 24 25  0  0  0]]


In [12]:
# Reverse-decode a sequence back to words to verify correctness
idx_to_word_t3 = {v: k for k, v in word_index_t3.items()}

print("=== Sequence → Decoded Words (verification) ===\n")
for seq in sequences_t3:
    decoded = [idx_to_word_t3.get(i, "<OOV>") for i in seq]
    print(f"  {seq}  →  {decoded}")

=== Sequence → Decoded Words (verification) ===

  [3, 4, 5, 6, 7]  →  ['i', 'love', 'natural', 'language', 'processing']
  [8, 9, 10, 11, 12]  →  ['deep', 'learning', 'models', 'are', 'powerful']
  [2, 13, 14, 15, 16, 17, 2, 18, 19]  →  ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
  [20, 21, 22, 23, 24, 25]  →  ['tensorflow', 'makes', 'building', 'neural', 'networks', 'easy']


---
### `Mini Predictive Text Application`

Features:
- **Tkinter GUI** — input box, Predict button, results panel
- **Confidence scores** displayed for each prediction
- **Top-5 predictions** shown as a ranked list with a progress bar per confidence
- Uses the LSTM model trained in Practical Task 1
- Larger extended corpus for richer vocabulary

In [13]:
extended_corpus = corpus + [
    "the weather is sunny and warm today",
    "the rain falls softly on the roof",
    "the wind blows strongly through the trees",
    "the flowers bloom in the spring garden",
    "the snow covers the mountains in winter",
    "python is a popular programming language",
    "data science combines statistics and programming",
    "artificial intelligence is transforming the world",
    "the internet connects people around the world",
    "computers process information very quickly",
    "the doctor examines the patient carefully",
    "the engineer designs bridges and buildings",
    "the artist paints beautiful pictures on canvas",
    "the musician plays the guitar every evening",
    "the chef cooks delicious meals in the kitchen",
    "the runner trains every morning before breakfast",
    "the students study hard for their exams",
    "the library contains thousands of books",
    "the scientist conducts experiments in the laboratory",
    "the pilot flies the airplane across the ocean",
    "she loves reading books in the evening",
    "he enjoys playing chess with his friends",
    "they went to the market to buy fresh vegetables",
    "the children played happily in the garden",
    "the team won the championship after hard work",
    "knowledge is the most powerful tool we have",
    "learning new skills opens many doors in life",
    "practice makes perfect in any field of study",
    "the brain processes language in remarkable ways",
    "words carry meaning that shapes our understanding",
]

print(f"Extended corpus size: {len(extended_corpus)} sentences")

Extended corpus size: 50 sentences


In [14]:
adv_tokenizer = Tokenizer()
adv_tokenizer.fit_on_texts(extended_corpus)
adv_vocab_size = len(adv_tokenizer.word_index) + 1

adv_sequences = []
for sentence in extended_corpus:
    tokens = adv_tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(tokens)):
        adv_sequences.append(tokens[:i + 1])

adv_max_len = max(len(s) for s in adv_sequences)
adv_padded = pad_sequences(adv_sequences, maxlen=adv_max_len, padding='pre')
adv_X = adv_padded[:, :-1]
adv_y = to_categorical(adv_padded[:, -1], num_classes=adv_vocab_size)

print(f"Vocabulary: {adv_vocab_size} tokens")
print(f"Training samples: {len(adv_sequences)}")
print(f"Max sequence length: {adv_max_len}")

Vocabulary: 201 tokens
Training samples: 289
Max sequence length: 9


In [15]:
tf.random.set_seed(42)

adv_model = Sequential([
    Embedding(input_dim=adv_vocab_size, output_dim=64, input_length=adv_max_len - 1),
    LSTM(256, return_sequences=True),
    LSTM(128),
    Dense(adv_vocab_size, activation='softmax'),
])
adv_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

adv_history = adv_model.fit(adv_X, adv_y, epochs=300, verbose=0)

for epoch in [0, 99, 199, 299]:
    print(f"Epoch {epoch+1:3d} — loss: {adv_history.history['loss'][epoch]:.4f}  "
          f"accuracy: {adv_history.history['accuracy'][epoch]:.4f}")

adv_idx_to_word = {v: k for k, v in adv_tokenizer.word_index.items()}

Epoch   1 — loss: 5.2957  accuracy: 0.0796
Epoch 100 — loss: 1.4184  accuracy: 0.7128
Epoch 200 — loss: 0.5820  accuracy: 0.8685
Epoch 300 — loss: 0.4561  accuracy: 0.8685


### `Predictive Text Application`

In [16]:
def adv_predict_next_word(seed_text, top_n=5):
    tokens = adv_tokenizer.texts_to_sequences([seed_text.lower()])[0]
    if not tokens:
        return []
    padded = pad_sequences([tokens], maxlen=adv_max_len - 1, padding='pre')
    probs = adv_model.predict(padded, verbose=0)[0]
    top_indices = np.argsort(probs)[::-1][:top_n]
    return [(adv_idx_to_word.get(i, '?'), float(probs[i])) for i in top_indices if i != 0]


BAR_MAX = 30

demo_seeds = [
    "the cat",
    "deep learning",
    "natural language",
    "the student loves",
    "artificial intelligence is",
    "the scientist",
    "practice makes",
    "knowledge is",
]

print("   PREDICTIVE TEXT APP — Confidence Report")
print("-" * 62)

for seed in demo_seeds:
    preds = adv_predict_next_word(seed, top_n=5)
    print(f"\n  Seed: \"{seed}\"")
    print(f"  {'Rank':<5} {'Word':<20} {'Confidence':>10}  Bar")
    print(f"  {'-'*4} {'-'*20} {'-'*10}  {'-'*BAR_MAX}")
    for rank, (word, conf) in enumerate(preds, 1):
        bar = "█" * int(BAR_MAX * conf)
        print(f"  #{rank:<4} {word:<20} {conf:>9.2%}  {bar}")

print("\n" + "-" * 62)

   PREDICTIVE TEXT APP — Confidence Report
--------------------------------------------------------------

  Seed: "the cat"
  Rank  Word                 Confidence  Bar
  ---- -------------------- ----------  ------------------------------
  #1    chased                  32.57%  █████████
  #2    sat                     32.42%  █████████
  #3    ran                     31.28%  █████████
  #4    bloom                    1.21%  
  #5    flew                     0.70%  

  Seed: "deep learning"
  Rank  Word                 Confidence  Bar
  ---- -------------------- ----------  ------------------------------
  #1    is                      98.96%  █████████████████████████████
  #2    a                        0.19%  
  #3    to                       0.18%  
  #4    new                      0.12%  
  #5    covers                   0.07%  

  Seed: "natural language"
  Rank  Word                 Confidence  Bar
  ---- -------------------- ----------  ------------------------------
  #1    

In [ ]:
import tkinter as tk
from tkinter import ttk, font as tkfont

def get_top_predictions(seed_text, top_n=5):
    tokens = adv_tokenizer.texts_to_sequences([seed_text.lower()])[0]
    if not tokens:
        return []
    padded = pad_sequences([tokens], maxlen=adv_max_len - 1, padding='pre')
    probs = adv_model.predict(padded, verbose=0)[0]
    top_indices = np.argsort(probs)[::-1][:top_n]
    return [(adv_idx_to_word.get(i, '?'), float(probs[i])) for i in top_indices if i != 0]


class PredictiveTextApp(tk.Tk):
    BAR_W = 220

    def __init__(self):
        super().__init__()
        self.title("Predictive Text — NLP Week 7")
        self.resizable(False, False)
        self.configure(bg="#1e1e2e")
        self._build_ui()

    def _build_ui(self):
        HEADER_BG  = "#313244"
        BODY_BG    = "#1e1e2e"
        ACCENT     = "#cba6f7"
        TEXT_COLOR = "#cdd6f4"
        BAR_FG     = "#a6e3a1"
        BAR_BG     = "#313244"
        BTN_BG     = "#cba6f7"
        BTN_FG     = "#1e1e2e"

        hdr = tk.Frame(self, bg=HEADER_BG, pady=12)
        hdr.pack(fill="x")
        tk.Label(hdr, text="Predictive Text App",
                 bg=HEADER_BG, fg=ACCENT,
                 font=("Segoe UI", 16, "bold")).pack()
        tk.Label(hdr, text="LSTM-powered next-word prediction",
                 bg=HEADER_BG, fg="#6c7086",
                 font=("Segoe UI", 9)).pack()

        inp_frame = tk.Frame(self, bg=BODY_BG, padx=20, pady=14)
        inp_frame.pack(fill="x")

        tk.Label(inp_frame, text="Enter a sentence fragment:",
                 bg=BODY_BG, fg=TEXT_COLOR,
                 font=("Segoe UI", 10)).pack(anchor="w")

        entry_frame = tk.Frame(inp_frame, bg=BODY_BG)
        entry_frame.pack(fill="x", pady=(4, 0))

        self.entry = tk.Entry(entry_frame, font=("Segoe UI", 12),
                              bg="#313244", fg=TEXT_COLOR,
                              insertbackground=TEXT_COLOR,
                              relief="flat", bd=0, width=36)
        self.entry.pack(side="left", ipady=6, padx=(0, 8))
        self.entry.insert(0, "the cat")
        self.entry.bind("<Return>", lambda e: self._predict())

        tk.Button(entry_frame, text="Predict ▶",
                  bg=BTN_BG, fg=BTN_FG,
                  font=("Segoe UI", 10, "bold"),
                  relief="flat", bd=0, padx=12, pady=6,
                  cursor="hand2",
                  command=self._predict).pack(side="left")

        res_frame = tk.Frame(self, bg=BODY_BG, padx=20, pady=4)
        res_frame.pack(fill="both", expand=True)

        tk.Label(res_frame, text="Top-5 Predictions",
                 bg=BODY_BG, fg="#6c7086",
                 font=("Segoe UI", 9, "italic")).pack(anchor="w")

        self.result_rows = []
        for rank in range(5):
            row = tk.Frame(res_frame, bg=BODY_BG, pady=5)
            row.pack(fill="x")

            rank_lbl = tk.Label(row, text=f"#{rank+1}",
                                bg=BODY_BG, fg="#6c7086",
                                font=("Consolas", 10), width=3, anchor="e")
            rank_lbl.pack(side="left", padx=(0, 8))

            word_lbl = tk.Label(row, text="—",
                                bg=BODY_BG, fg=TEXT_COLOR,
                                font=("Segoe UI", 11, "bold"), width=18, anchor="w")
            word_lbl.pack(side="left")

            bar_canvas = tk.Canvas(row, width=self.BAR_W, height=14,
                                   bg=BAR_BG, highlightthickness=0)
            bar_canvas.pack(side="left", padx=(4, 8))

            conf_lbl = tk.Label(row, text="",
                                bg=BODY_BG, fg=ACCENT,
                                font=("Consolas", 10), width=7, anchor="w")
            conf_lbl.pack(side="left")

            self.result_rows.append((word_lbl, bar_canvas, conf_lbl))

        self.status_var = tk.StringVar(value="Ready — type a phrase and press Predict")
        tk.Label(self, textvariable=self.status_var,
                 bg=HEADER_BG, fg="#6c7086",
                 font=("Segoe UI", 8), anchor="w", padx=10, pady=4).pack(fill="x", side="bottom")

        BAR_FG_COLOR = BAR_FG
        self._bar_fg = BAR_FG_COLOR

    def _predict(self):
        seed = self.entry.get().strip()
        if not seed:
            self.status_var.set("Please enter some text first.")
            return

        preds = get_top_predictions(seed, top_n=5)
        if not preds:
            self.status_var.set("No predictions — word(s) not in vocabulary.")
            for word_lbl, bar_canvas, conf_lbl in self.result_rows:
                word_lbl.config(text="—")
                bar_canvas.delete("all")
                conf_lbl.config(text="")
            return

        for rank, (word_lbl, bar_canvas, conf_lbl) in enumerate(self.result_rows):
            if rank < len(preds):
                word, conf = preds[rank]
                word_lbl.config(text=word)
                conf_lbl.config(text=f"{conf:.2%}")
                bar_canvas.delete("all")
                filled = int(self.BAR_W * conf)
                if filled > 0:
                    bar_canvas.create_rectangle(0, 0, filled, 14,
                                                fill=self._bar_fg, outline="")
            else:
                word_lbl.config(text="—")
                bar_canvas.delete("all")
                conf_lbl.config(text="")

        top_word, top_conf = preds[0]
        self.status_var.set(f'Best prediction for "{seed}": "{top_word}" ({top_conf:.2%} confidence)')


app = PredictiveTextApp()
app.mainloop()